<!-- TARGET: public course-materials repo, sessions/m06/lab.ipynb. Copy to work/m06/ before editing. -->
# Module 6 · Lab: five questions, five charts

**Python for Data Science · ESADE MIBA · Wed 18 Nov 2026**

Copy this notebook to `work/m06/lab.ipynb` and work on the copy. About 40 minutes.

The course project asks: *do late deliveries hurt review scores, and where should operations
act first?* Each question below is one piece of the answer, and each answer is a chart. Every
chart needs a **title that says the finding** and **axis labels in words**.

Fill in each cell with `...`, then run the **check** cell below it. Silence means correct; an
`AssertionError` tells you what to look at.

Every chart today starts from the **order mart** you built in the M05 lab:
`data/mart/orders_enriched.parquet`, one row per order. If the cell below stops with an
`AssertionError`, copy `sessions/m05/lab-solution.ipynb` to `work/m05/`, Run All, and come back.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

RAW = Path("../../data/raw")
MART_FILE = Path("../../data/mart/orders_enriched.parquet")

assert MART_FILE.exists(), (
    "No order mart yet. Copy sessions/m05/lab-solution.ipynb to work/m05/, Run All, then come back."
)
mart = pd.read_parquet(MART_FILE)
assert mart.shape == (99441, 13), (
    f"Your mart has shape {mart.shape}, expected (99441, 13). Run sessions/m05/lab-solution.ipynb to rebuild it."
)

delivered = mart[(mart["order_status"] == "delivered") & mart["is_late"].notna()]
delivered = delivered.assign(delivery=delivered["is_late"].map({True: "late", False: "on time"}))
print(len(delivered), "delivered orders with a delivery date")


def check_labelled(fig, *column_names):
    assert fig.layout.title.text, "Give the chart a title that says the finding."
    axis_titles = [fig.layout.xaxis.title.text, fig.layout.yaxis.title.text]
    for name in column_names:
        assert name not in axis_titles, f"Axis still shows the column name {name!r}: use labels=."

`delivered` holds the delivered orders that have a delivery date, with a `delivery` column
(`"late"` / `"on time"`). Questions 2–5 use it. `check_labelled` is used by the check cells.

## Q1 · Is the business growing? (5 min)

Count orders per `purchase_month` in `mart` (all statuses), from `"2017-01"` to `"2018-08"`.
Plot them as a **line** in `fig_monthly`.

In [ ]:
monthly = ...
fig_monthly = ...
fig_monthly

In [ ]:
assert fig_monthly.data[0].mode == "lines", "Use px.line."
assert len(fig_monthly.data[0].x) == 20, "Expected the 20 months from 2017-01 to 2018-08."
assert sum(fig_monthly.data[0].y) == mart["purchase_month"].between("2017-01", "2018-08").sum()
check_labelled(fig_monthly, "purchase_month", "orders")

## Q2 · How long do deliveries take? (5 min)

Plot the distribution of `delivery_days` for `delivered` orders in `fig_delivery_days`. Pick
the chart type for "how is one number spread out?". Zoom to 0–60 days and say so in the title.

In [ ]:
fig_delivery_days = ...
fig_delivery_days

In [ ]:
assert fig_delivery_days.data[0].type == "histogram", "How is one number spread out? A histogram."
assert len(fig_delivery_days.data[0].x) == len(delivered), "Plot every delivered order."
assert fig_delivery_days.layout.title.text, "Give the chart a title."
assert fig_delivery_days.layout.xaxis.title.text != "delivery_days", "Label the x axis in words."

## Q3 · Where are deliveries late most often? (10 min)

Build `late_by_state`: one row per `customer_state`, with `orders` and `late_share` (the share
of `delivered` orders that were late). Plot `late_share` as a bar chart in `fig_late_by_state`,
**sorted from highest to lowest**, y axis as a percentage.

In [ ]:
late_by_state = ...
fig_late_by_state = ...
fig_late_by_state

In [ ]:
plotted = pd.Series(list(fig_late_by_state.data[0].y), index=list(fig_late_by_state.data[0].x))
assert len(plotted) == 27, "One bar per state."
assert list(plotted) == sorted(plotted, reverse=True), "Sort the bars from highest to lowest."
assert plotted.index[0] == "AL"
check_labelled(fig_late_by_state, "customer_state", "late_share")

**Check the chart.** Recompute the late share per state from `delivered` with a fresh `groupby`
into `expected_late_share` (a Series indexed by state), then `assert` that the plotted bars
match it.

In [ ]:
expected_late_share = ...

In [ ]:
assert len(expected_late_share) == 27
assert ((plotted - expected_late_share).abs() < 1e-9).all()

## Q4 · Do late orders get worse reviews? (10 min)

Mean `review_score` for late vs on-time orders, as a bar chart in `fig_score_gap`, with the
number of orders in each group in the tooltip. Store the table in `score_gap`.

In [ ]:
score_gap = ...
fig_score_gap = ...
fig_score_gap

In [ ]:
plotted_scores = dict(zip(fig_score_gap.data[0].x, fig_score_gap.data[0].y))
assert set(plotted_scores) == {"late", "on time"}
assert round(plotted_scores["on time"], 2) == 4.29
assert round(plotted_scores["late"], 2) == 2.27
check_labelled(fig_score_gap, "delivery", "mean_score")

A mean hides the shape. Is every late order a bit worse, or are some a lot worse? Q5 answers it.

## Q5 · How are review scores spread, late vs on time? (10 min, finish at home if needed)

Orders without a review don't count here. Build `score_shares`: one row per `delivery` ×
`review_score`, with `orders` and `share` (the share of that delivery group's reviewed orders:
use `transform`). Plot it in `fig_score_shares` as a bar chart: `review_score` on x, `share` on
y, one colour per `delivery`, bars side by side (`barmode="group"`).

In [ ]:
score_shares = ...
fig_score_shares = ...
fig_score_shares

In [ ]:
assert len(score_shares) == 10, "One row per delivery group × score (2 × 5)."
assert (score_shares.groupby("delivery")["share"].sum().round(6) == 1).all(), "Shares must add up to 1 per group."
one_star_late = score_shares[(score_shares["delivery"] == "late") & (score_shares["review_score"] == 1)]
assert round(one_star_late["share"].iloc[0], 3) == 0.538
assert len(fig_score_shares.data) == 2, "One colour per delivery group."
check_labelled(fig_score_shares, "review_score", "share")

## For the project

In two sentences, for the operations team: what do your five charts say, and which state would
you look at first?

*Your answer:* ...

## Done?

Next week: charts that mislead, maps, and checking a chart with an `assert` that you hand in.